# 1 — Build SQLite Database

This notebook creates a local SQLite database for the project:
- Loads the LendingClub accepted-loans CSV
- Executes the SQL schema
- Populates the `raw_loans` table
- Creates the `clean_loans` analytics table

In [1]:
import sqlite3
from pathlib import Path

DB_PATH = Path("credit_risk.db")
con = sqlite3.connect(DB_PATH)

# confirm raw_loans exists
con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='raw_loans';").fetchall()

[('raw_loans',)]

In [3]:
import pandas as pd

CSV_PATH = Path("data/raw/accepted_2007_to_2018Q4.csv")

usecols = [
    "id","member_id","loan_amnt","funded_amnt","funded_amnt_inv","term","int_rate","installment",
    "grade","sub_grade","emp_title","emp_length","home_ownership","annual_inc","verification_status",
    "issue_d","loan_status","purpose","title","zip_code","addr_state","dti","delinq_2yrs",
    "earliest_cr_line","fico_range_low","fico_range_high","inq_last_6mths","open_acc","pub_rec",
    "revol_bal","revol_util","total_acc","initial_list_status","out_prncp","out_prncp_inv",
    "total_pymnt","total_pymnt_inv","total_rec_prncp","total_rec_int","total_rec_late_fee",
    "recoveries","collection_recovery_fee","last_pymnt_d","last_pymnt_amnt","next_pymnt_d"
]
con.execute("DELETE FROM raw_loans;")
con.commit()

chunksize = 50_000
total = 0

for i, chunk in enumerate(pd.read_csv(CSV_PATH, usecols=usecols, low_memory=False, chunksize=chunksize)):
    chunk.to_sql("raw_loans", con, if_exists="append", index=False)
    total += len(chunk)
    if (i + 1) % 5 == 0:
        print(f"Loaded {total:,} rows...")

print("Finished. Total rows loaded:", total)

Loaded 250,000 rows...
Loaded 500,000 rows...
Loaded 750,000 rows...
Loaded 1,000,000 rows...
Loaded 1,250,000 rows...
Loaded 1,500,000 rows...
Loaded 1,750,000 rows...
Loaded 2,000,000 rows...
Loaded 2,250,000 rows...
Finished. Total rows loaded: 2260701


In [4]:
import pandas as pd

pd.read_sql_query("SELECT COUNT(*) AS n_raw FROM raw_loans;", con)

,n_raw
0,2260701


In [5]:
con.execute("DROP TABLE IF EXISTS clean_loans;")

con.execute("""
CREATE TABLE clean_loans AS
SELECT
  row_id,
  id,
  member_id,
  issue_d,
  addr_state,
  purpose,
  grade,
  sub_grade,
  term,
  CAST(loan_amnt AS REAL) AS loan_amnt,
  CAST(funded_amnt AS REAL) AS funded_amnt,
  CAST(funded_amnt_inv AS REAL) AS funded_amnt_inv,
  CAST(REPLACE(int_rate, '%', '') AS REAL) AS int_rate_pct,
  CAST(installment AS REAL) AS installment,
  emp_length,
  home_ownership,
  CAST(annual_inc AS REAL) AS annual_inc,
  verification_status,
  CAST(dti AS REAL) AS dti,
  CAST(delinq_2yrs AS INTEGER) AS delinq_2yrs,
  earliest_cr_line,
  CAST(fico_range_low AS INTEGER) AS fico_low,
  CAST(fico_range_high AS INTEGER) AS fico_high,
  CAST(inq_last_6mths AS INTEGER) AS inq_last_6mths,
  CAST(open_acc AS INTEGER) AS open_acc,
  CAST(pub_rec AS INTEGER) AS pub_rec,
  CAST(revol_bal AS REAL) AS revol_bal,
  CAST(REPLACE(revol_util, '%', '') AS REAL) AS revol_util_pct,
  CAST(total_acc AS INTEGER) AS total_acc,
  initial_list_status,
  CAST(out_prncp AS REAL) AS out_prncp,
  CAST(out_prncp_inv AS REAL) AS out_prncp_inv,
  CAST(total_pymnt AS REAL) AS total_pymnt,
  CAST(total_pymnt_inv AS REAL) AS total_pymnt_inv,
  CAST(total_rec_prncp AS REAL) AS total_rec_prncp,
  CAST(total_rec_int AS REAL) AS total_rec_int,
  CAST(total_rec_late_fee AS REAL) AS total_rec_late_fee,
  CAST(recoveries AS REAL) AS recoveries,
  CAST(collection_recovery_fee AS REAL) AS collection_recovery_fee,
  last_pymnt_d,
  CAST(last_pymnt_amnt AS REAL) AS last_pymnt_amnt,
  loan_status,
  CASE
    WHEN loan_status IN ('Charged Off', 'Default') THEN 1
    WHEN loan_status = 'Fully Paid' THEN 0
    ELSE NULL
  END AS default_flag
FROM raw_loans
WHERE loan_status IN ('Charged Off', 'Default', 'Fully Paid');
""")

con.commit()
print("clean_loans created")

clean_loans created


In [6]:
pd.read_sql_query("""
SELECT
  COUNT(*) AS n_clean,
  SUM(default_flag) AS n_default,
  ROUND(100.0 * AVG(default_flag), 2) AS default_rate_pct
FROM clean_loans;
""", con)

,n_clean,n_default,default_rate_pct
0,1345350,268599,19.96
